# Quranic Arabic ASR for Microcontrollers
## Optimized for ESP32-S3 / STM32H7 / Embedded Devices

This notebook trains a **tiny ASR model** specifically designed for microcontrollers.

**Why NOT Whisper for MCUs:**
- Whisper-tiny = 39M params (~150MB) - too large for most MCUs
- Requires autoregressive decoding - complex for embedded
- High memory footprint during inference

**Our approach:**
- Custom tiny CNN+RNN model (~200K params, ~500KB INT8)
- CTC decoding - simple, no autoregressive loop
- TensorFlow Lite INT8 quantization
- Optimized for 256KB-1MB RAM devices

**Target platforms:**
| Platform | RAM | Flash | Status |
|----------|-----|-------|--------|
| ESP32-S3 | 512KB | 8MB | ✓ Supported |
| STM32H7 | 1MB | 2MB | ✓ Supported |
| nRF5340 | 512KB | 1MB | ✓ Supported |
| Raspberry Pi Pico | 264KB | 2MB | Edge case |

## 1. Setup

In [ ]:
# Install minimal dependencies
!pip install -q torch torchaudio datasets librosa soundfile
!pip install -q tensorflow
!pip cache purge

import os
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache"

!df -h /content

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 2. Define Arabic Character Set for Quran

Limited vocabulary optimized for Quranic Arabic text.

In [ ]:
# Quranic Arabic character set (optimized for MCU - minimal vocab)
ARABIC_CHARS = [
    '<blank>',  # CTC blank token (index 0)
    '<space>',  # Space
    'ا', 'أ', 'إ', 'آ', 'ٱ',  # Alef variants
    'ب', 'ت', 'ث', 'ج', 'ح', 'خ',
    'د', 'ذ', 'ر', 'ز', 'س', 'ش',
    'ص', 'ض', 'ط', 'ظ', 'ع', 'غ',
    'ف', 'ق', 'ك', 'ل', 'م', 'ن',
    'ه', 'ة', 'و', 'ؤ', 'ي', 'ئ', 'ى',
    # Diacritics (tashkeel) - essential for Quran
    'َ', 'ُ', 'ِ',  # Fatha, Damma, Kasra
    'ً', 'ٌ', 'ٍ',  # Tanween
    'ْ', 'ّ', 'ٰ',  # Sukun, Shadda, Superscript Alef
    'ۖ', 'ۗ', 'ۘ', 'ۙ', 'ۚ', 'ۛ', 'ۜ',  # Quranic stops
]

# Create mappings
char_to_idx = {c: i for i, c in enumerate(ARABIC_CHARS)}
idx_to_char = {i: c for i, c in enumerate(ARABIC_CHARS)}
VOCAB_SIZE = len(ARABIC_CHARS)

print(f"Vocabulary size: {VOCAB_SIZE} characters")
print(f"Characters: {ARABIC_CHARS}")

In [ ]:
def text_to_indices(text):
    """Convert Arabic text to indices."""
    indices = []
    for char in text:
        if char == ' ':
            indices.append(char_to_idx['<space>'])
        elif char in char_to_idx:
            indices.append(char_to_idx[char])
        # Skip unknown characters
    return indices

def indices_to_text(indices):
    """Convert indices back to text (CTC decode)."""
    text = []
    prev_idx = -1
    for idx in indices:
        if idx == 0:  # Blank
            prev_idx = -1
            continue
        if idx != prev_idx:
            char = idx_to_char.get(idx, '')
            if char == '<space>':
                text.append(' ')
            elif char != '<blank>':
                text.append(char)
        prev_idx = idx
    return ''.join(text)

# Test
test_text = "بِسْمِ اللَّهِ"
encoded = text_to_indices(test_text)
print(f"Test: '{test_text}' -> {encoded}")

## 3. Tiny ASR Model for MCUs

Architecture optimized for embedded deployment:
- **Depthwise separable convolutions** (fewer params)
- **Single-direction GRU** (half the params of BiGRU)
- **CTC output** (simple greedy decoding on MCU)
- **40 mel bins** (reduced from 80)

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """Efficient convolution for MCUs."""
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, kernel_size, stride, 
                                    padding=kernel_size//2, groups=in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1)
        self.bn = nn.BatchNorm2d(out_ch)
        
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return F.relu(x)


class TinyQuranASR(nn.Module):
    """
    Ultra-lightweight ASR for microcontrollers.
    Target: ~200K params -> ~500KB INT8
    """
    def __init__(self, n_mels=40, n_classes=VOCAB_SIZE, hidden_size=128):
        super().__init__()
        self.n_mels = n_mels
        
        # Lightweight CNN encoder with depthwise separable convs
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.conv2 = DepthwiseSeparableConv(32, 64, stride=2)
        self.conv3 = DepthwiseSeparableConv(64, 128, stride=2)
        self.conv4 = DepthwiseSeparableConv(128, hidden_size, stride=1)
        
        # Pool mel dimension
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        
        # Single-direction GRU (smaller than BiGRU)
        self.rnn = nn.GRU(hidden_size, hidden_size, num_layers=2, 
                          batch_first=True, dropout=0.1)
        
        # CTC output
        self.fc = nn.Linear(hidden_size, n_classes)
        
    def forward(self, x):
        # x: (batch, time, n_mels)
        x = x.unsqueeze(1)  # (batch, 1, time, n_mels)
        
        # CNN feature extraction
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)  # (batch, hidden, time/8, mels/8)
        
        # Collapse mel dimension
        x = self.pool(x)  # (batch, hidden, 1, time/8)
        x = x.squeeze(2).permute(0, 2, 1)  # (batch, time/8, hidden)
        
        # RNN sequence modeling
        x, _ = self.rnn(x)
        
        # Output logits for CTC
        x = self.fc(x)  # (batch, time, n_classes)
        return x


# Create model and check size
model = TinyQuranASR(n_mels=40, n_classes=VOCAB_SIZE, hidden_size=128)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Estimated FP32 size: {total_params * 4 / (1024**2):.2f} MB")
print(f"Estimated INT8 size: {total_params / (1024**2):.2f} MB")

# Test forward pass
dummy = torch.randn(1, 100, 40).to(device)
out = model(dummy)
print(f"\nInput shape: {dummy.shape}")
print(f"Output shape: {out.shape}  (time reduced by ~8x)")

## 4. Load and Preprocess Dataset

In [ ]:
import librosa
from datasets import load_dataset

# Audio parameters (optimized for MCU)
SAMPLE_RATE = 16000
N_MELS = 40  # Reduced from 80
N_FFT = 512  # Smaller FFT
HOP_LENGTH = 160  # 10ms hop
MAX_AUDIO_SEC = 10  # Limit audio length for MCU memory

def extract_mel_features(audio_array, sr):
    """Extract mel spectrogram features."""
    # Resample if needed
    if sr != SAMPLE_RATE:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLE_RATE)
    
    # Truncate to max length
    max_samples = MAX_AUDIO_SEC * SAMPLE_RATE
    if len(audio_array) > max_samples:
        audio_array = audio_array[:max_samples]
    
    # Extract mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=audio_array,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )
    
    # Log mel
    log_mel = librosa.power_to_db(mel, ref=np.max)
    
    # Normalize
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-8)
    
    return log_mel.T  # (time, n_mels)

print(f"Audio config: {SAMPLE_RATE}Hz, {N_MELS} mels, {N_FFT} FFT, {HOP_LENGTH} hop")

In [ ]:
# Load dataset with streaming
print("Loading Quran dataset...")

dataset = load_dataset(
    "rabah2026/Quran-Ayah-Corpus",
    streaming=True,
    trust_remote_code=True
)

# Inspect structure
sample = next(iter(dataset['train']))
print(f"\nDataset keys: {sample.keys()}")
for k, v in sample.items():
    if k != 'audio':
        print(f"  {k}: {v}")

In [ ]:
# Preprocess dataset
TRAIN_SAMPLES = 3000  # Limit for disk space (increase if possible)

print(f"Processing {TRAIN_SAMPLES} samples...")

train_data = []
skipped = 0

for i, sample in enumerate(dataset['train'].take(TRAIN_SAMPLES + 500)):
    if len(train_data) >= TRAIN_SAMPLES:
        break
        
    try:
        # Get audio
        audio = sample['audio']
        mel = extract_mel_features(audio['array'], audio['sampling_rate'])
        
        # Get text (try different field names)
        text = sample.get('text') or sample.get('transcription') or \
               sample.get('ayah_text') or sample.get('sentence', '')
        
        if not text or len(text) < 3:
            skipped += 1
            continue
        
        # Encode text
        labels = text_to_indices(text)
        if len(labels) < 2:
            skipped += 1
            continue
        
        train_data.append({
            'mel': mel,
            'labels': labels,
            'text': text
        })
        
        if len(train_data) % 500 == 0:
            print(f"  Processed {len(train_data)} samples...")
            clear_memory()
            
    except Exception as e:
        skipped += 1
        continue

print(f"\nTotal samples: {len(train_data)}, Skipped: {skipped}")

# Split
split_idx = int(len(train_data) * 0.9)
train_set = train_data[:split_idx]
val_set = train_data[split_idx:]
print(f"Train: {len(train_set)}, Validation: {len(val_set)}")

## 5. Training with CTC Loss

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class QuranDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        return (
            torch.FloatTensor(item['mel']),
            torch.LongTensor(item['labels'])
        )

def collate_fn(batch):
    """Pad sequences for batching."""
    mels, labels = zip(*batch)
    
    # Pad mels
    mel_lengths = torch.LongTensor([m.size(0) for m in mels])
    mels_padded = pad_sequence(mels, batch_first=True)
    
    # Pad labels
    label_lengths = torch.LongTensor([l.size(0) for l in labels])
    labels_padded = pad_sequence(labels, batch_first=True)
    
    return mels_padded, labels_padded, mel_lengths, label_lengths

# Create dataloaders
BATCH_SIZE = 16

train_loader = DataLoader(
    QuranDataset(train_set),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    QuranDataset(val_set),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Training setup
EPOCHS = 50
LEARNING_RATE = 1e-3

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    
    for mels, labels, mel_lens, label_lens in loader:
        mels = mels.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward
        logits = model(mels)  # (batch, time, classes)
        log_probs = F.log_softmax(logits, dim=-1)
        
        # CTC expects (time, batch, classes)
        log_probs = log_probs.permute(1, 0, 2)
        
        # Output lengths (time is reduced by ~8x due to strided convs)
        output_lens = mel_lens // 8
        output_lens = torch.clamp(output_lens, min=1)
        
        # CTC loss
        loss = criterion(log_probs, labels, output_lens, label_lens)
        
        if not torch.isnan(loss) and not torch.isinf(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
    
    return total_loss / len(loader)

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for mels, labels, mel_lens, label_lens in loader:
            mels = mels.to(device)
            labels = labels.to(device)
            
            logits = model(mels)
            log_probs = F.log_softmax(logits, dim=-1).permute(1, 0, 2)
            
            output_lens = torch.clamp(mel_lens // 8, min=1)
            loss = criterion(log_probs, labels, output_lens, label_lens)
            
            if not torch.isnan(loss):
                total_loss += loss.item()
    
    return total_loss / len(loader)

print("Training setup complete!")

In [ ]:
# Training loop
print("Starting training...")
print("="*50)

best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 10

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss = validate(model, val_loader, criterion)
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_quran_asr_mcu.pt")
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break
    
    # Clear memory periodically
    if (epoch + 1) % 10 == 0:
        clear_memory()

print("\n" + "="*50)
print(f"Training complete! Best val loss: {best_val_loss:.4f}")

# Load best model
model.load_state_dict(torch.load("best_quran_asr_mcu.pt"))

## 6. Test Inference

In [ ]:
def greedy_decode(logits):
    """Simple greedy CTC decoding (MCU-friendly)."""
    # Get most likely class at each timestep
    pred_ids = torch.argmax(logits, dim=-1)
    return indices_to_text(pred_ids.cpu().numpy())

# Test on validation samples
model.eval()
print("Sample predictions:\n")

with torch.no_grad():
    for i in range(min(5, len(val_set))):
        sample = val_set[i]
        mel = torch.FloatTensor(sample['mel']).unsqueeze(0).to(device)
        
        logits = model(mel)[0]
        predicted = greedy_decode(logits)
        
        print(f"Expected:  {sample['text']}")
        print(f"Predicted: {predicted}")
        print("-" * 50)

## 7. Export to TensorFlow Lite INT8

In [ ]:
import tensorflow as tf

# First export to ONNX, then to TF SavedModel, then to TFLite
print("Step 1: Export PyTorch to ONNX...")

model.eval()
model_cpu = model.cpu()

# Fixed input shape for MCU (max 5 seconds = 500 frames at 10ms hop)
MAX_FRAMES = 500
dummy_input = torch.randn(1, MAX_FRAMES, N_MELS)

ONNX_PATH = "quran_asr_mcu.onnx"

torch.onnx.export(
    model_cpu,
    dummy_input,
    ONNX_PATH,
    input_names=["mel_input"],
    output_names=["logits"],
    opset_version=12,
    # Fixed shape for MCU (no dynamic axes)
)

print(f"✓ ONNX model saved: {ONNX_PATH}")
print(f"  Input shape: (1, {MAX_FRAMES}, {N_MELS})")

In [ ]:
# Convert ONNX to TensorFlow SavedModel
!pip install -q onnx-tf onnx

print("\nStep 2: Convert ONNX to TensorFlow...")

import onnx
from onnx_tf.backend import prepare

onnx_model = onnx.load(ONNX_PATH)
tf_rep = prepare(onnx_model)
tf_rep.export_graph("quran_asr_tf")

print("✓ TensorFlow SavedModel created: quran_asr_tf/")

In [ ]:
# Convert to TFLite with INT8 quantization
print("\nStep 3: Convert to TFLite INT8...")

def representative_dataset():
    """Generate calibration data for INT8 quantization."""
    for i in range(min(100, len(train_set))):
        mel = train_set[i]['mel']
        # Pad/truncate to fixed size
        if len(mel) > MAX_FRAMES:
            mel = mel[:MAX_FRAMES]
        else:
            mel = np.pad(mel, ((0, MAX_FRAMES - len(mel)), (0, 0)))
        yield [mel.astype(np.float32).reshape(1, MAX_FRAMES, N_MELS)]

# Load SavedModel
converter = tf.lite.TFLiteConverter.from_saved_model("quran_asr_tf")

# INT8 quantization settings
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# Convert
tflite_model = converter.convert()

# Save
TFLITE_PATH = "quran_asr_mcu_int8.tflite"
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

model_size_kb = len(tflite_model) / 1024
print(f"\n✓ TFLite INT8 model saved: {TFLITE_PATH}")
print(f"  Model size: {model_size_kb:.2f} KB")
print(f"  Input: INT8 ({1}, {MAX_FRAMES}, {N_MELS})")
print(f"  Output: INT8 logits")

In [ ]:
# Verify TFLite model
print("\nVerifying TFLite model...")

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
print(f"  Shape: {input_details[0]['shape']}")
print(f"  Type: {input_details[0]['dtype']}")
print(f"  Quantization: {input_details[0]['quantization']}")

print("\nOutput details:")
print(f"  Shape: {output_details[0]['shape']}")
print(f"  Type: {output_details[0]['dtype']}")

# Test inference
test_mel = train_set[0]['mel']
if len(test_mel) > MAX_FRAMES:
    test_mel = test_mel[:MAX_FRAMES]
else:
    test_mel = np.pad(test_mel, ((0, MAX_FRAMES - len(test_mel)), (0, 0)))

# Quantize input
input_scale, input_zero_point = input_details[0]['quantization']
test_input = (test_mel / input_scale + input_zero_point).astype(np.int8)
test_input = test_input.reshape(1, MAX_FRAMES, N_MELS)

interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]['index'])
print(f"\n✓ Inference successful! Output shape: {output.shape}")

## 8. Generate MCU Deployment Files

In [ ]:
# Convert TFLite to C array for embedding in firmware
print("Converting TFLite to C header...")

!xxd -i {TFLITE_PATH} > quran_asr_model.h

# Fix variable name in header
with open("quran_asr_model.h", "r") as f:
    content = f.read()

content = content.replace("quran_asr_mcu_int8_tflite", "g_quran_asr_model")
content = content.replace("quran_asr_mcu_int8_tflite_len", "g_quran_asr_model_len")

# Add header guard
header = """#ifndef QURAN_ASR_MODEL_H_
#define QURAN_ASR_MODEL_H_

// Quranic Arabic ASR Model for MCU
// Generated by Quranic_ASR_MCU.ipynb
// Input: INT8 mel spectrogram (1, 500, 40)
// Output: INT8 logits for CTC decoding

"""

footer = "\n#endif  // QURAN_ASR_MODEL_H_\n"

with open("quran_asr_model.h", "w") as f:
    f.write(header + content + footer)

print("✓ C header saved: quran_asr_model.h")

In [ ]:
# Generate complete MCU inference code
mcu_code = f'''// =============================================================
// Quranic Arabic ASR - MCU Inference Code
// For ESP32-S3 / STM32H7 / TFLite Micro compatible devices
// =============================================================

#include "tensorflow/lite/micro/micro_interpreter.h"
#include "tensorflow/lite/micro/micro_mutable_op_resolver.h"
#include "tensorflow/lite/schema/schema_generated.h"
#include "quran_asr_model.h"

// Model constants (must match training)
constexpr int kMaxFrames = {MAX_FRAMES};  // 5 seconds of audio
constexpr int kNumMels = {N_MELS};
constexpr int kSampleRate = {SAMPLE_RATE};
constexpr int kHopLength = {HOP_LENGTH};
constexpr int kNumClasses = {VOCAB_SIZE};

// Tensor arena - adjust based on model
constexpr int kTensorArenaSize = 150 * 1024;  // ~150KB
alignas(16) uint8_t tensor_arena[kTensorArenaSize];

// Character mapping for CTC decoding
const char* kArabicChars[] = {{
    "",     // blank (0)
    " ",    // space (1)
    "ا", "أ", "إ", "آ", "ٱ",
    "ب", "ت", "ث", "ج", "ح", "خ",
    "د", "ذ", "ر", "ز", "س", "ش",
    "ص", "ض", "ط", "ظ", "ع", "غ",
    "ف", "ق", "ك", "ل", "م", "ن",
    "ه", "ة", "و", "ؤ", "ي", "ئ", "ى",
    "\xD9\x8E", "\xD9\x8F", "\xD9\x90",  // Fatha, Damma, Kasra
    "\xD9\x8B", "\xD9\x8C", "\xD9\x8D",  // Tanween
    "\xD9\x92", "\xD9\x91", "\xD9\xB0",  // Sukun, Shadda, Superscript Alef
}};

class QuranASR {{
public:
    bool Init() {{
        model_ = tflite::GetModel(g_quran_asr_model);
        if (model_->version() != TFLITE_SCHEMA_VERSION) {{
            return false;
        }}
        
        // Add required ops for the model
        static tflite::MicroMutableOpResolver<12> resolver;
        resolver.AddConv2D();
        resolver.AddDepthwiseConv2D();
        resolver.AddFullyConnected();
        resolver.AddRelu();
        resolver.AddReshape();
        resolver.AddMean();
        resolver.AddQuantize();
        resolver.AddDequantize();
        resolver.AddTranspose();
        resolver.AddUnidirectionalSequenceGRU();  // For GRU
        
        static tflite::MicroInterpreter static_interpreter(
            model_, resolver, tensor_arena, kTensorArenaSize);
        interpreter_ = &static_interpreter;
        
        if (interpreter_->AllocateTensors() != kTfLiteOk) {{
            return false;
        }}
        
        input_ = interpreter_->input(0);
        output_ = interpreter_->output(0);
        return true;
    }}
    
    // Run inference on mel spectrogram
    // mel_data: INT8 quantized mel spectrogram (kMaxFrames x kNumMels)
    // result: output buffer for decoded text
    bool Transcribe(const int8_t* mel_data, char* result, int result_size) {{
        // Copy input
        memcpy(input_->data.int8, mel_data, kMaxFrames * kNumMels);
        
        // Run inference
        if (interpreter_->Invoke() != kTfLiteOk) {{
            return false;
        }}
        
        // CTC greedy decode
        int output_len = output_->dims->data[1];  // Time dimension
        int prev_idx = -1;
        int result_pos = 0;
        
        for (int t = 0; t < output_len && result_pos < result_size - 4; t++) {{
            // Find max class
            int max_idx = 0;
            int8_t max_val = output_->data.int8[t * kNumClasses];
            
            for (int c = 1; c < kNumClasses; c++) {{
                int8_t val = output_->data.int8[t * kNumClasses + c];
                if (val > max_val) {{
                    max_val = val;
                    max_idx = c;
                }}
            }}
            
            // CTC: skip blanks and repeats
            if (max_idx != 0 && max_idx != prev_idx) {{
                const char* ch = kArabicChars[max_idx];
                int ch_len = strlen(ch);
                memcpy(result + result_pos, ch, ch_len);
                result_pos += ch_len;
            }}
            prev_idx = max_idx;
        }}
        
        result[result_pos] = '\\0';
        return true;
    }}

private:
    const tflite::Model* model_ = nullptr;
    tflite::MicroInterpreter* interpreter_ = nullptr;
    TfLiteTensor* input_ = nullptr;
    TfLiteTensor* output_ = nullptr;
}};

// ========== Mel Spectrogram Computation ==========
// For MCU: Use CMSIS-DSP or fixed-point FFT library
// This is a placeholder - implement based on your platform

#include <arm_math.h>  // For CMSIS-DSP (STM32)

void ComputeMelSpectrogram(const int16_t* audio, int audio_len,
                           int8_t* mel_output, float input_scale, int8_t input_zero_point) {{
    // 1. Apply pre-emphasis filter
    // 2. Frame audio into {HOP_LENGTH}-sample hops
    // 3. Apply Hanning window
    // 4. Compute FFT using CMSIS-DSP arm_rfft_q15
    // 5. Apply mel filterbank ({N_MELS} filters)
    // 6. Log compression
    // 7. Normalize and quantize to INT8
    
    // TODO: Implement for your specific platform
    // Consider using existing libraries:
    // - ESP32: ESP-DSP
    // - STM32: CMSIS-DSP
    // - Generic: KissFFT
}}

// ========== Example Usage ==========
/*
QuranASR asr;
char transcription[512];

void setup() {{
    if (!asr.Init()) {{
        // Handle error
    }}
}}

void loop() {{
    // Record audio (16kHz, mono, 5 seconds max)
    int16_t audio_buffer[{SAMPLE_RATE} * 5];
    int audio_len = RecordAudio(audio_buffer);
    
    // Compute mel spectrogram
    int8_t mel_buffer[{MAX_FRAMES} * {N_MELS}];
    ComputeMelSpectrogram(audio_buffer, audio_len, mel_buffer, input_scale, input_zp);
    
    // Run ASR
    if (asr.Transcribe(mel_buffer, transcription, sizeof(transcription))) {{
        Serial.println(transcription);
    }}
}}
*/
'''

with open("quran_asr_mcu.cpp", "w") as f:
    f.write(mcu_code)

print("✓ MCU inference code saved: quran_asr_mcu.cpp")

In [ ]:
# Generate config header
config_header = f'''#ifndef QURAN_ASR_CONFIG_H_
#define QURAN_ASR_CONFIG_H_

// =============================================================
// Quranic Arabic ASR - Configuration
// =============================================================

// Audio input parameters
#define QURAN_ASR_SAMPLE_RATE    {SAMPLE_RATE}
#define QURAN_ASR_MAX_AUDIO_SEC  5
#define QURAN_ASR_MAX_SAMPLES    ({SAMPLE_RATE} * QURAN_ASR_MAX_AUDIO_SEC)

// Mel spectrogram parameters
#define QURAN_ASR_N_MELS         {N_MELS}
#define QURAN_ASR_N_FFT          {N_FFT}
#define QURAN_ASR_HOP_LENGTH     {HOP_LENGTH}
#define QURAN_ASR_MAX_FRAMES     {MAX_FRAMES}

// Model parameters
#define QURAN_ASR_NUM_CLASSES    {VOCAB_SIZE}
#define QURAN_ASR_TENSOR_ARENA   (150 * 1024)  // 150KB

// Quantization parameters (from TFLite model)
#define QURAN_ASR_INPUT_SCALE    {input_scale:.6f}f
#define QURAN_ASR_INPUT_ZP       {input_zero_point}

#endif  // QURAN_ASR_CONFIG_H_
'''

with open("quran_asr_config.h", "w") as f:
    f.write(config_header)

print("✓ Config header saved: quran_asr_config.h")

## 9. Save to Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/quran-asr-mcu"
os.makedirs(SAVE_PATH, exist_ok=True)

# Copy all deployment files
files_to_save = [
    "quran_asr_mcu_int8.tflite",  # TFLite model
    "quran_asr_model.h",           # Model as C array
    "quran_asr_mcu.cpp",           # Inference code
    "quran_asr_config.h",          # Configuration
    "best_quran_asr_mcu.pt",       # PyTorch weights (backup)
]

for f in files_to_save:
    if os.path.exists(f):
        shutil.copy(f, SAVE_PATH)
        print(f"✓ Saved: {f}")

print(f"\nAll files saved to: {SAVE_PATH}")

## 10. Deployment Summary

**Files generated:**
| File | Purpose |
|------|---------|
| `quran_asr_mcu_int8.tflite` | TFLite INT8 model (~500KB) |
| `quran_asr_model.h` | Model as C array for embedding |
| `quran_asr_mcu.cpp` | Inference code template |
| `quran_asr_config.h` | Configuration parameters |

**To deploy on your MCU:**

1. **ESP32-S3 (ESP-IDF):**
   ```bash
   idf.py add-dependency "espressif/esp-tflite-micro"
   ```
   Copy `quran_asr_model.h`, `quran_asr_mcu.cpp`, `quran_asr_config.h` to your project.

2. **STM32 (STM32CubeIDE):**
   - Enable X-CUBE-AI or TFLite Micro
   - Add CMSIS-DSP for mel computation
   - Copy generated files

3. **Arduino (using TFLite Micro library):**
   - Install `Arduino_TensorFlowLite` library
   - Use provided template

**Memory requirements:**
- Model size: ~500KB Flash
- Tensor arena: ~150KB RAM
- Audio buffer: ~160KB RAM (5 sec @ 16kHz)
- **Total RAM needed: ~320KB minimum**

In [ ]:
# Final summary
print("="*60)
print("QURANIC ARABIC ASR FOR MICROCONTROLLERS - COMPLETE")
print("="*60)
print(f"\nModel: TinyQuranASR")
print(f"Parameters: {total_params:,}")
print(f"Model size (INT8): {model_size_kb:.2f} KB")
print(f"\nAudio config:")
print(f"  Sample rate: {SAMPLE_RATE} Hz")
print(f"  Max duration: 5 seconds")
print(f"  Mel bins: {N_MELS}")
print(f"\nTarget platforms:")
print(f"  ✓ ESP32-S3 (512KB RAM)")
print(f"  ✓ STM32H7 (1MB RAM)")
print(f"  ✓ nRF5340 (512KB RAM)")
print(f"\nFiles saved to Google Drive: {SAVE_PATH}")
print("="*60)